In [25]:
import pyreadr
import pandas as pd
import re
import xml
from bs4 import BeautifulSoup
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, classification_report, confusion_matrix
import numpy as np

example_preds = pyreadr.read_r('../../results/example-preds.RData')
raw_claims = pyreadr.read_r('../../data/claims-raw.RData')
raw_claims_df = raw_claims['claims_raw']
raw_claims_df['text_tmp'].head(5)


0    <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 S...
1    <!DOCTYPE html>\n<html lang="en">\n<head>\n   ...
2    <!DOCTYPE html>\n<html>\n<head>\n<meta charset...
3    <!DOCTYPE HTML>\n<html lang="en">\n<head>\n\t<...
4    <!DOCTYPE html>\n<html>\n<head>\n<meta charset...
Name: text_tmp, dtype: str

In [2]:
def parse_fn(html):
    soup = BeautifulSoup(html, features = "xml")
    try:
        for script in soup(["script", "style"]):
            script.decompose() # destroy script and style elements

        # get head and paragraph text
        ph_text = ' '.join(spoon.get_text() for spoon in soup.select('h1, p'))

        # remove urls and emails
            # \S@\S+ means non-whitespace characters followed by @ then followed by 1 or more non-whitespace characters
            # pattern = r'https?://\S+|www\.\S+' which means http with optional s followed by :// and then 1 or more non-whitespace characters
            # |www\.\S+ means or www. then 1 or more non-whitespace characters
        text = re.sub(r'\S@\S+|https?://\S+', "", ph_text)

        # remove apostraphes

        text = re.sub("'", "",text)

        # remove non-alphabetic characters and nbsp
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text) # add space between camel case words

        text = text.lower()

        text = re.sub(r'\s+', ' ', text).strip() # collapse multiple spaces into one and strip leading/trailing whitespace
        # remove punctuation and newlines
        
        return text 
    except Exception as err:
        print(f"Error parsing HTML: {err}")
        return None


def parse_data(df):
    df = df[df['text_tmp'].str.contains('<!', regex = False)].copy() # filter for rows that contain html tags
    df['text_clean'] = df['text_tmp'].apply(parse_fn) # apply parse_fn and create a new column
    return df

clean_claims_df = parse_data(raw_claims_df)


In [44]:
# Split the data into training and testing sets
train_df, test_df = train_test_split(clean_claims_df, test_size = 0.2, random_state = 101422)

# tokenizer and lemmatizer

lemmatizer = WordNetLemmatizer()

# strip punctuation from stop words and lemmatize
stpwrd = {w.replace("'", "") for w in ENGLISH_STOP_WORDS}

def tokenize(text):
    tokens = text.split()
    return [lemmatizer.lemmatize(token) for token in tokens if token not in stpwrd]

def projection_fn (dtm, prop, max_components = 500):
    # create truncated svd object
    
    tsvd = TruncatedSVD(n_components = max_components, random_state = 101422)
    # fit model, learning the weights of components to dtm
    tsvd.fit(dtm)

    # explained variance ratio relative to the total variance
    # get cumsum to explain the prop of variance explained by components
    cum = tsvd.explained_variance_ratio_.cumsum()
    n_pc = int(np.argmax(cum >= prop) + 1) # get the first index where cum >= prop and add 1 to compensate for numpy 0-indexing 

    return tsvd, n_pc, cum

# Create DTM with tf-idf
vectorizer = TfidfVectorizer(tokenizer = tokenize, lowercase = False, token_pattern = None)
# fit_transform fits model on data and reduces dimensionality
# transform just reduces dimensionality
train_dtm = vectorizer.fit_transform(train_df['text_clean']) # fit vectorizer on training data and transform it into a dt matrix
test_dtm = vectorizer.transform(test_df['text_clean']) # transform test data into a dt matrix using vectorizer

train_labels = train_df[['.id', 'bclass', 'mclass']]
test_labels = test_df[['.id', 'bclass', 'mclass']]

### transform the dtm and create the truncated SVD

tsvd, n_pc, cum = projection_fn(train_dtm, prop = 0.7)
# use transform to multiply the dtm by the first n_pc columns of V in SVD to turn the dtm into a shorter matrix with n_pc scores
x_train = tsvd.transform(train_dtm)[:, :n_pc]  # reproject training dtm onto first n_pc columns 
x_test = tsvd.transform(test_dtm)[:, :n_pc] # same but for test dtm

# convert y levels to binary and ensure correct factoring since in R glm does this automatically with family = binomial
y_train = (pd.Series(train_labels['bclass']) == 'Relevant claim content').astype(int) # convert to binary labels so that relevant = 1 and irrelevant = 0
y_test = (pd.Series(test_labels['bclass']) == 'Relevant claim content').astype(int) # same with test labels
    


In [50]:
#### Fit the logistic regression model

claimsf = LogisticRegression(max_iter = 1000)
# Need to learn LogisticRegression documentation :/

claimsf.fit(x_train, y_train)
# predict_proba(x) gives probability estimates. Why is it spelled like that??
probs = claimsf.predict_proba(x_test)[:,1]
preds = (probs > 0.5).astype(int)


In [56]:
## Metrics

print(f"Accuracy: {accuracy_score(y_test, preds)}")
print(f"AUC: {roc_auc_score(y_test, preds)}")
print(f"Sensitivity: {recall_score(y_test, preds)}")
print(f"Specificity: {recall_score(y_test, preds, pos_label = 0)}")
print(f"Confusion matrix: \n {confusion_matrix(y_test, preds)}")
print(f"\n Classification report: \n {classification_report(y_test, preds)}")

Accuracy: 0.6401869158878505
AUC: 0.6390435637078283
Sensitivity: 0.39436619718309857
Specificity: 0.8837209302325582
Confusion matrix: 
 [[190  25]
 [129  84]]

 Classification report: 
               precision    recall  f1-score   support

           0       0.60      0.88      0.71       215
           1       0.77      0.39      0.52       213

    accuracy                           0.64       428
   macro avg       0.68      0.64      0.62       428
weighted avg       0.68      0.64      0.62       428



In [48]:

print(f"No. of components to explain 70% of variance: {n_pc}")
print(f"Cum explained variance of first {n_pc} components: {cum[n_pc - 1]}")
print(x_test)

No. of components to explain 70% of variance: 251
Cum explained variance of first 251 components: 0.7006987351081301
[[ 7.63689712e-03  7.18912704e-02  8.37609659e-02 ... -8.36747504e-03
   1.63763021e-03 -6.00707058e-03]
 [ 2.01107068e-03  1.71224721e-02  3.53449300e-02 ...  4.82531114e-03
  -1.44822657e-03  3.86866839e-03]
 [ 1.79818240e-03  1.86094162e-02  1.36686829e-02 ... -2.88456676e-04
  -2.47428373e-03  1.15377245e-03]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 1.56906922e-02  1.25522611e-01  5.15953605e-01 ...  1.04635157e-03
   8.75523988e-03 -3.43665196e-02]
 [ 9.96204945e-01 -8.50617798e-02 -1.33727200e-02 ...  7.07737992e-05
  -5.30053729e-05 -3.20416898e-04]]
